In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# This command removes everything inside the /content/drive directory
!rm -r /content/drive/*

In [9]:
# ====================================================================
# 🔹 TEST DATA - BLOCK 1: SETUP, FUNCTIONS, AND INITIAL PARSING
# ====================================================================

# --- Step 1.1: Installs and Imports ---
!pip install -q pandas numpy scikit-learn sentence_transformers torch torchvision tqdm joblib
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
import torch
from PIL import Image
import requests
from io import BytesIO
from tqdm.auto import tqdm
import joblib
import os
import time

# --- Step 1.2: Path Definitions ---
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
MODELS_PATH = '/content/drive/My Drive/ML_Price_Prediction/models/'
os.makedirs(MODELS_PATH, exist_ok=True)

# --- Define paths for TEST data files ---
TEST_PARSED_PATH = os.path.join(BASE_PATH, 'test_parsed.csv')
IMG_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'test_image_embeddings_v2_checkpoint.csv')
TEXT_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'test_text_embeddings_v2_checkpoint.csv')
TEST_FINAL_CLEANED_PATH = os.path.join(BASE_PATH, 'test_final_cleaned_v2.csv')

# --- Step 1.3: Function Definitions (Identical to training) ---
def parse_product_details_v5(text):
    num_of_packs, net_qty_value, unit = 1, 1.0, 'count'
    text_lower = str(text).lower()
    pack_patterns = [
        r'\(pack of (\d+)\)', r'pack of (\d+)', r'(\d+)\s*per case', r'(\d+)\s*count',
        r'(\d+)\s*ct', r'(\d+)\s*pack', r'(\d+)-pack', r'(\d+)-ct', r'(\d+)\s*pcs'
    ]
    found_packs = [1]
    for pattern in pack_patterns:
        matches = re.findall(pattern, text_lower)
        if matches:
            for match in matches:
                found_packs.append(int(match))
    num_of_packs = max(found_packs)
    qty_pattern = r'(\d+\.?\d*)\s*-?\s*(fluid ounce|fl oz|ounces|ounce|oz|grams|gram|g|pounds|pound|lbs|lb|liters|liter|l|milliliters|milliliter|ml)\b'
    qty_match = re.search(qty_pattern, text_lower)
    if qty_match:
        net_qty_value = float(qty_match.group(1))
        unit_str = qty_match.group(2)
        if 'fl' in unit_str or 'fluid' in unit_str: unit = 'fl_oz'
        elif 'oz' in unit_str or 'ounce' in unit_str: unit = 'ounce'
        elif 'g' in unit_str or 'gram' in unit_str: unit = 'gram'
        elif 'lb' in unit_str or 'pound' in unit_str: unit = 'pound'
        elif 'l' in unit_str or 'liter' in unit_str: unit = 'liter'
        elif 'ml' in unit_str or 'milliliter' in unit_str: unit = 'ml'
    return num_of_packs, net_qty_value, unit

def extract_item_and_brand(text):
    item_name_match = re.search(r"Item Name:\s*(.*)", text)
    if item_name_match:
        item_name = item_name_match.group(1).strip()
        brand = item_name.split(' ')[0]
        return item_name, brand
    return "Unknown", "Unknown"

# --- Step 1.4: Execution on Test Data ---
print("\n--- Running Initial Parsing on Test Data ---")
df_test = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
df_test['catalog_content'] = df_test['catalog_content'].fillna('')
df_test[['item_name', 'brand']] = df_test['catalog_content'].apply(lambda x: pd.Series(extract_item_and_brand(x)))
df_test[['num_of_packs', 'net_qty', 'unit']] = df_test['catalog_content'].apply(lambda x: pd.Series(parse_product_details_v5(x)))
df_test.to_csv(TEST_PARSED_PATH, index=False)

print(f"\n✅ Initial parsing of test data complete. Parsed data saved to:\n{TEST_PARSED_PATH}")
display(df_test.head())


--- Running Initial Parsing on Test Data ---

✅ Initial parsing of test data complete. Parsed data saved to:
/content/drive/My Drive/ML_Price_Prediction/dataset/test_parsed.csv


,sample_id,catalog_content,image_link,item_name,brand,num_of_packs,net_qty,unit
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...,Rani 14-Spice Eshamaya's Mango Chutney (Indian...,Rani,1,10.5,ounce
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...,Natural MILK TEA Flavoring extract by HALO PAN...,Natural,1,2.0,ounce
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...,Honey,1,2.0,pound
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,https://m.media-amazon.com/images/I/51Ex6uOH7y...,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2),Vlasic,2,16.0,ounce
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",https://m.media-amazon.com/images/I/71QYlrOMoS...,"McCormick Culinary Vanilla Extract, 32 fl oz -...",McCormick,1,32.0,fl_oz


In [10]:
# ====================================================================
# 🔹 TEST DATA - BLOCK 2: IMAGE EMBEDDING GENERATION (RESTART-PROOF)
# ====================================================================
import pandas as pd
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import torch
from torchvision import models, transforms
from tqdm.auto import tqdm
import os

print("\n--- Generating Image Embeddings for Test Data ---")

# --- Define Paths ---
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
TEST_PARSED_PATH = os.path.join(BASE_PATH, 'test_parsed.csv')
IMG_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'test_image_embeddings_v2_checkpoint.csv')

# Load the parsed test data
df_parsed = pd.read_csv(TEST_PARSED_PATH)

# Define the image embedding function
def get_image_embedding(image_url, model, preprocess, device):
    try:
        response = requests.get(image_url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        batch_t = torch.unsqueeze(preprocess(img), 0).to(device)
        with torch.no_grad():
            embedding = model(batch_t)
        return embedding.cpu().numpy().flatten()
    except Exception:
        return np.zeros(576)

df_img_embeds = pd.DataFrame()
if os.path.exists(IMG_EMBED_CHECKPOINT_PATH) and len(pd.read_csv(IMG_EMBED_CHECKPOINT_PATH)) == len(df_parsed):
    print("✅ Full image embedding file for test data found. Skipping.")
else:
    print("⚠️ Full image embedding file not found. Generating embeddings (this will take several hours)...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    img_model = models.mobilenet_v3_small(pretrained=True)
    img_model.classifier = torch.nn.Identity()
    img_model.to(device); img_model.eval()
    preprocess = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

    processed_ids = set()
    if os.path.exists(IMG_EMBED_CHECKPOINT_PATH):
        print("   - Partial checkpoint found. Resuming...")
        df_img_embeds = pd.read_csv(IMG_EMBED_CHECKPOINT_PATH)
        processed_ids = set(df_img_embeds['sample_id'])

    df_to_process = df_parsed[~df_parsed['sample_id'].isin(processed_ids)]

    if not df_to_process.empty:
        new_embeddings = []
        # Process in chunks and save periodically
        SAVE_INTERVAL = 1000
        for i, (index, row) in enumerate(tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Generating Image Embeddings")):
            embedding = get_image_embedding(row['image_link'], img_model, preprocess, device)
            result = {'sample_id': row['sample_id'], **{f'img_embed_{j}': val for j, val in enumerate(embedding)}}
            new_embeddings.append(result)

            if (i + 1) % SAVE_INTERVAL == 0 or (i + 1) == len(df_to_process):
                df_new_chunk = pd.DataFrame(new_embeddings)
                df_img_embeds = pd.concat([df_img_embeds, df_new_chunk], ignore_index=True)
                df_img_embeds.to_csv(IMG_EMBED_CHECKPOINT_PATH, index=False)
                new_embeddings = [] # Clear the batch from memory
                print(f"\n💾 Image embedding checkpoint saved! Processed {len(df_img_embeds)} rows.")

    print("\n✅ Image embedding generation for test data is complete.")


--- Generating Image Embeddings for Test Data ---


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


⚠️ Full image embedding file not found. Generating embeddings (this will take several hours)...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 87.1MB/s]


Generating Image Embeddings:   0%|          | 0/75000 [00:00<?, ?it/s]


💾 Image embedding checkpoint saved! Processed 1000 rows.

💾 Image embedding checkpoint saved! Processed 2000 rows.

💾 Image embedding checkpoint saved! Processed 3000 rows.

💾 Image embedding checkpoint saved! Processed 4000 rows.

💾 Image embedding checkpoint saved! Processed 5000 rows.

💾 Image embedding checkpoint saved! Processed 6000 rows.

💾 Image embedding checkpoint saved! Processed 7000 rows.

💾 Image embedding checkpoint saved! Processed 8000 rows.

💾 Image embedding checkpoint saved! Processed 9000 rows.

💾 Image embedding checkpoint saved! Processed 10000 rows.

💾 Image embedding checkpoint saved! Processed 11000 rows.

💾 Image embedding checkpoint saved! Processed 12000 rows.

💾 Image embedding checkpoint saved! Processed 13000 rows.

💾 Image embedding checkpoint saved! Processed 14000 rows.

💾 Image embedding checkpoint saved! Processed 15000 rows.

💾 Image embedding checkpoint saved! Processed 16000 rows.

💾 Image embedding checkpoint saved! Processed 17000 rows.

💾 Ima

In [2]:
# ====================================================================
# 🔹 TEST DATA - BLOCK 3: TEXT EMBEDDING GENERATION (RESTART-PROOF)
# ====================================================================
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import os

print("\n--- Generating Text Embeddings for Test Data ---")

# --- Define Paths ---
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
TEST_PARSED_PATH = os.path.join(BASE_PATH, 'test_parsed.csv')
TEXT_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'test_text_embeddings_v2_checkpoint.csv')

# Load the parsed test data
df_parsed = pd.read_csv(TEST_PARSED_PATH)

df_text_embeds = pd.DataFrame()
if os.path.exists(TEXT_EMBED_CHECKPOINT_PATH) and len(pd.read_csv(TEXT_EMBED_CHECKPOINT_PATH)) == len(df_parsed):
    print("✅ Full text embedding file for test data found. Skipping.")
else:
    print("⚠️ Full text embedding file not found. Generating embeddings (this will take 1-2 hours)...")
    text_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

    processed_ids = set()
    if os.path.exists(TEXT_EMBED_CHECKPOINT_PATH):
        print("   - Partial checkpoint found. Resuming...")
        df_text_embeds = pd.read_csv(TEXT_EMBED_CHECKPOINT_PATH)
        processed_ids = set(df_text_embeds['sample_id'])

    df_to_process = df_parsed[~df_parsed['sample_id'].isin(processed_ids)]

    if not df_to_process.empty:
        texts_to_encode = df_to_process['catalog_content'].fillna('').tolist()
        text_embeddings = text_model.encode(texts_to_encode, show_progress_bar=True, batch_size=128)

        df_new_embeds = pd.DataFrame(text_embeddings, columns=[f'text_embed_{i}' for i in range(text_embeddings.shape[1])])
        df_new_embeds['sample_id'] = df_to_process['sample_id'].values

        df_text_embeds = pd.concat([df_text_embeds, df_new_embeds], ignore_index=True)
        df_text_embeds.to_csv(TEXT_EMBED_CHECKPOINT_PATH, index=False)
        print("\n✅ Text embedding generation complete and saved.")
    else:
        print("   - All text embeddings were already processed.")


--- Generating Text Embeddings for Test Data ---
⚠️ Full text embedding file not found. Generating embeddings (this will take 1-2 hours)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/586 [00:00<?, ?it/s]


✅ Text embedding generation complete and saved.


In [4]:
# ====================================================================
# 🔹 TEST DATA - BLOCK 4: FINAL ASSEMBLY AND CLEANING
# ====================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import os

print("\n--- Assembling and cleaning all test features ---")

# --- Step 4.1: Load all the test data components ---
BASE_PATH = '/content/drive/My Drive/ML_Price_Prediction/dataset/'
MODELS_PATH = '/content/drive/My Drive/ML_Price_Prediction/models/'
TEST_PARSED_PATH = os.path.join(BASE_PATH, 'test_parsed.csv')
IMG_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'test_image_embeddings_v2_checkpoint.csv')
TEXT_EMBED_CHECKPOINT_PATH = os.path.join(BASE_PATH, 'test_text_embeddings_v2_checkpoint.csv')
TEST_FINAL_CLEANED_PATH = os.path.join(BASE_PATH, 'test_final_cleaned_v2.csv')

df_parsed = pd.read_csv(TEST_PARSED_PATH)
df_img_embeds = pd.read_csv(IMG_EMBED_CHECKPOINT_PATH)
df_text_embeds = pd.read_csv(TEXT_EMBED_CHECKPOINT_PATH)

# Merge everything
df_final = pd.merge(df_parsed, df_img_embeds, on='sample_id', how='left')
df_final = pd.merge(df_final, df_text_embeds, on='sample_id', how='left')

df_final['total_qty'] = df_final['num_of_packs'] * df_final['net_qty']
df_final['is_qty_missing'] = ((df_final['num_of_packs'] == 1) & (df_final['net_qty'] == 1.0)).astype(int)

# --- Step 4.2: Load the "Recipe" (scaler and caps) from the Models Folder ---
print("\nLoading scaler and outlier caps from training process...")
SCALER_PATH = os.path.join(MODELS_PATH, 'scaler_v2.pkl')
OUTLIER_CAPS_PATH = os.path.join(MODELS_PATH, 'outlier_caps_v2.npy')
scaler = joblib.load(SCALER_PATH)
outlier_caps = np.load(OUTLIER_CAPS_PATH, allow_pickle=True).item()
print("✅ Rules loaded successfully.")

# --- Step 4.3: Apply the Recipe to the Test Data ---
# Apply Outlier Capping (using training rules)
for col, cap_value in outlier_caps.items():
    if col in df_final.columns:
        df_final.loc[df_final[col] > cap_value, col] = cap_value
print("✅ Outliers capped using training set thresholds.")

# Apply Scaling (using training rules)
num_cols_to_scale = ['num_of_packs', 'net_qty', 'total_qty']
# IMPORTANT: Use .transform() ONLY, NOT .fit_transform()
df_final[num_cols_to_scale] = scaler.transform(df_final[num_cols_to_scale])
print(f"✅ Scaling complete using training set parameters.")

# --- Step 4.4: Ensure Final Column Consistency ---
TRAIN_FINAL_CLEANED_PATH = os.path.join(BASE_PATH, 'train_final_cleaned_v2.csv')
df_train_cleaned = pd.read_csv(TRAIN_FINAL_CLEANED_PATH)
final_training_columns = [col for col in df_train_cleaned.columns if col not in ['price', 'log_price']]

# Reorder/select columns in the test set to match the training blueprint perfectly
df_test_final = df_final.reindex(columns=final_training_columns, fill_value=0)
print("✅ Column consistency with training set confirmed.")

# --- Step 4.5: Save the final, model-ready test file ---
df_test_final.to_csv(TEST_FINAL_CLEANED_PATH, index=False)
print(f"\n💾 Final clean test data saved to:\n{TEST_FINAL_CLEANED_PATH}")

print("\n\n🎉🎉🎉 SUCCESS! Your entire test data preprocessing is complete. 🎉🎉🎉")
display(df_test_final.head())


--- Assembling and cleaning all test features ---

Loading scaler and outlier caps from training process...
✅ Rules loaded successfully.
✅ Outliers capped using training set thresholds.
✅ Scaling complete using training set parameters.
✅ Column consistency with training set confirmed.

💾 Final clean test data saved to:
/content/drive/My Drive/ML_Price_Prediction/dataset/test_final_cleaned_v2.csv


🎉🎉🎉 SUCCESS! Your entire test data preprocessing is complete. 🎉🎉🎉


,sample_id,catalog_content,image_link,item_name,brand,num_of_packs,net_qty,unit,img_embed_0,img_embed_1,...,text_embed_376,text_embed_377,text_embed_378,text_embed_379,text_embed_380,text_embed_381,text_embed_382,text_embed_383,total_qty,is_qty_missing
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...,Rani 14-Spice Eshamaya's Mango Chutney (Indian...,Rani,-0.210489,-0.121297,ounce,0.795839,-0.073801,...,0.032901,-0.038549,0.102396,-0.002740,0.065963,-0.073983,0.003490,0.001306,-0.138561,0
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...,Natural MILK TEA Flavoring extract by HALO PAN...,Natural,-0.210489,-0.241725,ounce,0.217709,-0.077965,...,0.045323,-0.126972,0.003267,-0.092224,0.047413,-0.093157,0.102705,-0.012593,-0.152707,0
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...,Honey,-0.210489,-0.241725,pound,-0.203582,-0.021151,...,0.000268,-0.034375,0.067146,-0.003414,0.041320,-0.103453,-0.039478,-0.009965,-0.152707,0
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,https://m.media-amazon.com/images/I/51Ex6uOH7y...,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2),Vlasic,-0.184792,-0.043374,ounce,0.793961,0.065461,...,-0.040931,0.014969,-0.003423,-0.054427,0.062931,-0.102680,0.004074,0.041173,-0.102781,0
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",https://m.media-amazon.com/images/I/71QYlrOMoS...,"McCormick Culinary Vanilla Extract, 32 fl oz -...",McCormick,-0.210489,0.183313,fl_oz,0.569582,-0.140060,...,-0.024664,-0.030132,-0.023025,-0.123908,0.039280,-0.063152,0.041553,-0.094272,-0.102781,0
